# CS434 PA2 - Model A Baseline

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import torch

REPO_URL = "https://github.com/syeddaniyalg/cs434-pa2.git"
REPO_DIR = Path("/kaggle/working/cs434-pa2")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
print(Path.cwd())

## Hardware

In [ ]:
gpu_count = torch.cuda.device_count()
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {gpu_count}")
for index in range(gpu_count):
    print(f"GPU {index}: {torch.cuda.get_device_name(index)}")
if gpu_count < 2:
    raise RuntimeError("This notebook expects two Kaggle GPUs")

In [ ]:
DATA_DIR = REPO_DIR / "data"
CHECKPOINT_DIR = REPO_DIR / "checkpoints"
ARTIFACT_DIR = REPO_DIR / "artifacts"

def run(command):
    print(" ".join(map(str, command)), flush=True)
    subprocess.run(list(map(str, command)), check=True)

def wait_for(processes):
    failures = []
    for label, process in processes:
        return_code = process.wait()
        print(f"{label} finished with code {return_code}", flush=True)
        if return_code != 0:
            failures.append(label)
    if failures:
        raise RuntimeError(f"Failed runs: {failures}")

def train_pair(model):
    processes = []
    for seed, device in ((0, "cuda:0"), (1, "cuda:1")):
        command = [
            sys.executable, "-m", "src.train",
            "--model", model,
            "--seed", seed,
            "--device", device,
            "--data-dir", DATA_DIR,
            "--checkpoint-dir", CHECKPOINT_DIR,
            "--artifact-dir", ARTIFACT_DIR,
            "--workers", 4,
            "--resume",
        ]
        processes.append((f"{model} seed {seed}", subprocess.Popen(list(map(str, command)))))
    wait_for(processes)

def evaluate_pair(models, sym_models=()):
    processes = []
    for seed, device in ((0, "cuda:0"), (1, "cuda:1")):
        command = [
            sys.executable, "-m", "src.evaluate",
            "--models", *models,
            "--sym-models", *sym_models,
            "--seed", seed,
            "--device", device,
            "--data-dir", DATA_DIR,
            "--checkpoint-dir", CHECKPOINT_DIR,
            "--artifact-dir", ARTIFACT_DIR,
            "--workers", 4,
        ]
        command.append("--baselines" if seed == 0 else "--no-baselines")
        processes.append((f"evaluation seed {seed}", subprocess.Popen(list(map(str, command)))))
    wait_for(processes)

## Dataset and exact checks

In [ ]:
run([sys.executable, "-m", "src.data", "--root", DATA_DIR, "--download"])
run([sys.executable, "-m", "src.checks", "--artifact-dir", ARTIFACT_DIR])

## Figure 1 - training-set optical flow

In [ ]:
run([sys.executable, "-m", "src.optical_flow", "--data-dir", DATA_DIR, "--artifact-dir", ARTIFACT_DIR])

## Model A

In [ ]:
train_pair("A")
evaluate_pair(["A"])
run([sys.executable, "-m", "src.results", "--artifact-dir", ARTIFACT_DIR, "--stage", "baseline"])
run([sys.executable, "-m", "src.plotting", "--stage", "baseline", "--data-dir", DATA_DIR, "--checkpoint-dir", CHECKPOINT_DIR, "--artifact-dir", ARTIFACT_DIR, "--device", "cuda:0"])

## Export model A

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

model_a_archive = Path("/kaggle/working/cs434-pa2-model-a.zip")

with ZipFile(model_a_archive, "w", ZIP_DEFLATED) as archive:

    for folder in (CHECKPOINT_DIR, ARTIFACT_DIR):

        for path in folder.rglob("*"):

            if path.is_file():

                archive.write(path, path.relative_to(REPO_DIR))

print(model_a_archive)